# Phase 3 — Preference Fine-Tuning (DPO) — 5 Trials

**Base:** Best SFT model from `01_sft_training.ipynb`  
**Dataset:** `Intel/orca_dpo_pairs` (3,000 samples subset)  
**Method:** DPO + LoRA via TRL DPOTrainer

**Before running:**
1. Mount Google Drive (Cell 2).
2. Set `BEST_SFT_ADAPTER_PATH` in Cell 4 to the path printed at the end of `01_sft_training.ipynb`.
3. Upload `test_prompts.json` (with filled references) to session or Drive.
4. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.9.6 \
    bitsandbytes==0.43.3 accelerate==0.34.2 datasets==2.21.0 \
    evaluate==0.4.3 bert-score==0.3.13 sacrebleu==2.4.3

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import gc
import json
import os
import time

import evaluate
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import DPOTrainer, DPOConfig

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Cell 4: Config — UPDATE BEST_SFT_ADAPTER_PATH before running ──────────────
MODEL_ID               = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
BEST_SFT_ADAPTER_PATH  = "/content/drive/MyDrive/sft_trials/T2"  # ← UPDATE THIS
PROMPTS_PATH           = "test_prompts.json"
DRIVE_OUTPUT_DIR       = "/content/drive/MyDrive/dpo_trials"
DPO_RESULTS_PATH       = "/content/drive/MyDrive/dpo_results.csv"
BASELINE_CSV           = "/content/drive/MyDrive/baseline_results.csv"
SFT_RESULTS_CSV        = "/content/drive/MyDrive/sft_results.csv"
MAX_NEW_TOKENS         = 200
TEMPERATURE            = 0.7
TOP_P                  = 0.9
MAX_LENGTH             = 512   # max prompt+chosen/rejected length for DPO
DPO_DATASET_SIZE       = 3000

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Cell 5: Load test prompts ──────────────────────────────────────────────────
with open(PROMPTS_PATH) as f:
    data = json.load(f)

test_prompts    = [p["prompt"]    for p in data["prompts"]]
test_references = [p["reference"] for p in data["prompts"]]

empty = [i+1 for i, r in enumerate(test_references) if not r.strip()]
if empty:
    raise ValueError(f"Missing references for prompt IDs: {empty}")

sacrebleu = evaluate.load("sacrebleu")
print(f"Loaded {len(test_prompts)} test prompts.")

In [ ]:
# ── Cell 6: Load and preprocess Intel/orca_dpo_pairs ──────────────────────────
raw_dpo = load_dataset("Intel/orca_dpo_pairs", split="train")
print(f"Full dataset size: {len(raw_dpo)}")
print("Columns:", raw_dpo.column_names)
print("Sample:", raw_dpo[0])

In [ ]:
# ── Cell 7: Format DPO dataset ─────────────────────────────────────────────────
# Intel/orca_dpo_pairs columns: system, question, chosen, rejected
# DPOTrainer expects: prompt, chosen, rejected (all as plain strings)

def format_dpo(example):
    system = example.get("system", "").strip()
    question = example["question"].strip()
    if system:
        prompt = f"{system}\n\nUser: {question}\n\nAssistant:"
    else:
        prompt = f"User: {question}\n\nAssistant:"
    return {
        "prompt":   prompt,
        "chosen":   example["chosen"].strip(),
        "rejected": example["rejected"].strip(),
    }

dpo_ds = raw_dpo.shuffle(seed=42).select(range(DPO_DATASET_SIZE))
dpo_ds = dpo_ds.map(format_dpo, remove_columns=dpo_ds.column_names)

# Filter out examples where chosen == rejected (sometimes happens)
dpo_ds = dpo_ds.filter(lambda x: x["chosen"] != x["rejected"])

dpo_split = dpo_ds.train_test_split(test_size=0.1, seed=42)
dpo_train = dpo_split["train"]
dpo_val   = dpo_split["test"]
print(f"DPO Train: {len(dpo_train)}  |  Val: {len(dpo_val)}")
print("\nSample prompt:", dpo_train[0]["prompt"][:200])
print("Chosen[:100]:",   dpo_train[0]["chosen"][:100])
print("Rejected[:100]:", dpo_train[0]["rejected"][:100])

In [ ]:
# ── Cell 8: 4-bit BnB config ───────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
# ── Cell 9: Evaluation helper ──────────────────────────────────────────────────
def evaluate_model(model, tokenizer, prompts, references):
    model.eval()
    tokenizer.padding_side = "left"
    generated = []
    for prompt in prompts:
        formatted = (
            "Below is an instruction that describes a task. Write a response that "
            "appropriately completes the request.\n\n"
            f"### Instruction:\n{prompt}\n\n### Response:\n"
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        generated.append(text)

    bleu_scores = [
        sacrebleu.compute(predictions=[g], references=[[r]])["score"]
        for g, r in zip(generated, references)
    ]
    _, _, F1 = bert_score_fn(generated, references, lang="en",
                              rescale_with_baseline=True, verbose=False)
    bert_f1 = [f.item() for f in F1]

    return round(sum(bleu_scores)/len(bleu_scores), 4), \
           round(sum(bert_f1)/len(bert_f1), 4), \
           generated

In [ ]:
# ── Cell 10: DPO trial configurations ─────────────────────────────────────────
# LoRA rank/modules are inherited from the best SFT trial.
# Update `lora_target_modules` to match the best SFT trial's target_modules.

BEST_SFT_LORA_RANK    = 8      # ← UPDATE to match best SFT trial r
BEST_SFT_LORA_ALPHA   = 16     # ← UPDATE to match best SFT trial lora_alpha
BEST_SFT_LORA_MODULES = ["q_proj", "v_proj"]  # ← UPDATE to match best trial

DPO_TRIAL_CONFIGS = [
    {
        "name": "D1",
        "desc": "beta=0.1, lr=1e-5, bs=2, 1 epoch",
        "beta": 0.1,
        "lr": 1e-5,
        "batch_size": 2,
        "grad_accum": 4,
        "epochs": 1,
    },
    {
        "name": "D2",
        "desc": "beta=0.1, lr=5e-5, bs=4, 2 epochs",
        "beta": 0.1,
        "lr": 5e-5,
        "batch_size": 4,
        "grad_accum": 2,
        "epochs": 2,
    },
    {
        "name": "D3",
        "desc": "beta=0.3, lr=1e-5, bs=2, 1 epoch",
        "beta": 0.3,
        "lr": 1e-5,
        "batch_size": 2,
        "grad_accum": 4,
        "epochs": 1,
    },
    {
        "name": "D4",
        "desc": "beta=0.5, lr=2e-5, bs=4, 2 epochs",
        "beta": 0.5,
        "lr": 2e-5,
        "batch_size": 4,
        "grad_accum": 2,
        "epochs": 2,
    },
    {
        "name": "D5",
        "desc": "beta=0.1, lr=1e-4, bs=2, 1 epoch (high LR)",
        "beta": 0.1,
        "lr": 1e-4,
        "batch_size": 2,
        "grad_accum": 4,
        "epochs": 1,
    },
]

In [ ]:
# ── Cell 11: DPO training loop ─────────────────────────────────────────────────
# Each trial: load SFT adapter as policy model + frozen ref model → DPO train

dpo_results = []

for cfg in DPO_TRIAL_CONFIGS:
    trial_name = cfg["name"]
    print(f"\n{'='*70}")
    print(f"  DPO TRIAL {trial_name}: {cfg['desc']}")
    print(f"{'='*70}")

    # ── Load tokenizer ─────────────────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(BEST_SFT_ADAPTER_PATH)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # ── Load SFT model as policy (trainable) ───────────────────────────────────
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    policy_model = PeftModel.from_pretrained(base, BEST_SFT_ADAPTER_PATH, is_trainable=True)
    policy_model.config.use_cache = False

    # ── Load frozen reference model (SFT model, non-trainable) ────────────────
    base_ref = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    ref_model = PeftModel.from_pretrained(base_ref, BEST_SFT_ADAPTER_PATH, is_trainable=False)

    # ── New LoRA adapter on top of loaded SFT weights ──────────────────────────
    new_lora = LoraConfig(
        r=BEST_SFT_LORA_RANK,
        lora_alpha=BEST_SFT_LORA_ALPHA,
        target_modules=BEST_SFT_LORA_MODULES,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    output_dir = f"/content/dpo_{trial_name}"
    drive_dir  = f"{DRIVE_OUTPUT_DIR}/{trial_name}"

    # ── DPO config ─────────────────────────────────────────────────────────────
    dpo_config = DPOConfig(
        beta=cfg["beta"],
        output_dir=output_dir,
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        gradient_accumulation_steps=cfg["grad_accum"],
        per_device_eval_batch_size=2,
        learning_rate=cfg["lr"],
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="none",
        optim="paged_adamw_8bit",
        max_length=MAX_LENGTH,
        max_prompt_length=256,
        remove_unused_columns=False,
    )

    # ── DPOTrainer ─────────────────────────────────────────────────────────────
    trainer = DPOTrainer(
        model=policy_model,
        ref_model=ref_model,
        args=dpo_config,
        train_dataset=dpo_train,
        eval_dataset=dpo_val,
        tokenizer=tokenizer,
        peft_config=new_lora,
    )

    # ── Train ──────────────────────────────────────────────────────────────────
    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    # ── Val loss ───────────────────────────────────────────────────────────────
    eval_output = trainer.evaluate()
    val_loss = round(eval_output["eval_loss"], 4)

    # ── Save adapter ───────────────────────────────────────────────────────────
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    os.makedirs(drive_dir, exist_ok=True)
    !cp -r {output_dir}/* {drive_dir}/
    print(f"Adapter saved to {drive_dir}")

    # ── Evaluate on test prompts ───────────────────────────────────────────────
    print("Evaluating on 10 test prompts...")
    mean_bleu, mean_bert_f1, generated_texts = evaluate_model(
        trainer.model, tokenizer, test_prompts, test_references
    )

    dpo_results.append({
        "trial": trial_name,
        "desc": cfg["desc"],
        "beta": cfg["beta"],
        "lr": cfg["lr"],
        "batch_size": cfg["batch_size"],
        "grad_accum": cfg["grad_accum"],
        "epochs": cfg["epochs"],
        "val_loss": val_loss,
        "mean_bleu": mean_bleu,
        "mean_bert_f1": mean_bert_f1,
        "train_time_min": round(train_time / 60, 1),
        "adapter_path": drive_dir,
        "generated_texts": generated_texts,
    })

    print(f"\n  {trial_name} | val_loss={val_loss} | BLEU={mean_bleu} | BERTScore F1={mean_bert_f1} | time={train_time/60:.1f}min")

    # ── Free GPU memory ────────────────────────────────────────────────────────
    del trainer, policy_model, ref_model, base, base_ref
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll 5 DPO trials complete.")

In [ ]:
# ── Cell 12: DPO results table ─────────────────────────────────────────────────
cols = ["trial","desc","beta","lr","batch_size","epochs","val_loss","mean_bleu","mean_bert_f1","train_time_min"]
df_dpo = pd.DataFrame(dpo_results)[cols]
pd.set_option("display.max_colwidth", 50)
print(df_dpo.to_string(index=False))

df_dpo.to_csv(DPO_RESULTS_PATH, index=False)
print(f"\nSaved to {DPO_RESULTS_PATH}")

In [ ]:
# ── Cell 13: Select best DPO model ────────────────────────────────────────────
df_dpo_ranked = df_dpo.sort_values(
    by=["mean_bert_f1", "mean_bleu", "val_loss"],
    ascending=[False, False, True]
).reset_index(drop=True)

best_dpo = df_dpo_ranked.iloc[0]
print("\n" + "="*60)
print(f"  BEST DPO TRIAL: {best_dpo['trial']}")
print(f"  Description:    {best_dpo['desc']}")
print(f"  BERTScore F1:   {best_dpo['mean_bert_f1']}")
print(f"  BLEU:           {best_dpo['mean_bleu']}")
print(f"  Val Loss:       {best_dpo['val_loss']}")
print("="*60)

In [ ]:
# ── Cell 14: Three-way comparison — Base vs Best SFT vs Best DPO ──────────────
# Load saved CSVs (assumes baseline and SFT notebooks have already run)

try:
    df_baseline = pd.read_csv(BASELINE_CSV)
    df_sft      = pd.read_csv(SFT_RESULTS_CSV)

    best_sft_row  = df_sft.sort_values(by=["mean_bert_f1","mean_bleu","val_loss"],
                                        ascending=[False,False,True]).iloc[0]
    best_dpo_idx  = df_dpo_ranked.index[0]
    best_dpo_gen  = dpo_results[best_dpo_idx]["generated_texts"]

    print("\nTHREE-WAY METRIC COMPARISON")
    print("-" * 50)
    print(f"Base model       — BLEU: {df_baseline['bleu'].mean():.4f}  | BERTScore F1: {df_baseline['bert_f1'].mean():.4f}")
    print(f"Best SFT ({best_sft_row['trial']})   — BLEU: {best_sft_row['mean_bleu']:.4f}  | BERTScore F1: {best_sft_row['mean_bert_f1']:.4f}")
    print(f"Best DPO ({best_dpo['trial']})   — BLEU: {best_dpo['mean_bleu']:.4f}  | BERTScore F1: {best_dpo['mean_bert_f1']:.4f}")

except FileNotFoundError as e:
    print(f"Could not load comparison CSVs: {e}")
    print("Best DPO metrics:")
    print(f"  BLEU: {best_dpo['mean_bleu']}  |  BERTScore F1: {best_dpo['mean_bert_f1']}")

In [ ]:
# ── Cell 15: Print generated outputs from best DPO (for report) ───────────────
best_dpo_idx  = df_dpo_ranked.index[0]
best_dpo_gen  = dpo_results[best_dpo_idx]["generated_texts"]

print(f"Generated outputs from best DPO trial ({best_dpo['trial']}):")
for i, (prompt, gen, ref) in enumerate(
    zip(test_prompts, best_dpo_gen, test_references)
):
    print(f"\n[{i+1}] {prompt[:80]}")
    print(f"  DPO Output: {gen[:200]}")
    print(f"  Reference:  {ref[:200]}")

## Finalizing the Report

1. Copy the **three-way comparison table** (Cell 14 output) into **Section 6** of `report.md`.
2. Copy the DPO results table (Cell 12) into **Section 5 (DPO Experiments)**.
3. Add the best DPO generated outputs for 2-3 representative prompts as examples in **Section 7 (Analysis)**.
4. Once `report.md` is complete, convert it to `.docx` using the `/anthropic-skills:docx` skill in Claude Code.